In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
from datetime import datetime
SILVER_PATH = "/Volumes/medalliaon_replication/la_crime/la_crime/"

In [0]:
@dlt.table(
    name="la_crime_clean",
    comment="Cleaned LA crime Silver table"
)
def la_crime_clean():
    """
    Create the Silver (cleaned) LA crime table from the Bronze table.
    """

    # 🔹 Read from Bronze (choose one of these depending on your setup)
    # If bronze is a normal table:
    df = spark.table("workspace.default.la_crime_raw")
    # If bronze is a DLT table, you would use:
    # df = dlt.read("la_crime_raw")

    # =============================
    # CLEANING TRANSFORMATIONS
    # =============================

    # 1. Clean and parse dates
    df_cleaned = df \
        .withColumn(
            "date_reported_clean",
            to_date(col("Date_Rptd"), "yyyy MMM dd")
        ) \
        .withColumn(
            "date_occurred_clean",
            to_date(col("DATE_OCC"), "MM/dd/yyyy")
        )

    # 2. Clean and parse time (convert to proper time format)
    df_cleaned = df_cleaned \
        .withColumn(
            "time_occurred_clean",
            when(col("TIME_OCC").isNotNull(),
                 lpad(col("TIME_OCC"), 4, "0"))
            .otherwise(None)
        ) \
        .withColumn(
            "hour_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 1, 2).cast("int"))
            .otherwise(None)
        ) \
        .withColumn(
            "minute_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 3, 2).cast("int"))
            .otherwise(None)
        )

    # 3. Cast numeric fields
    df_cleaned = df_cleaned \
        .withColumn("dr_no_clean", col("DR_NO").cast("bigint")) \
        .withColumn("area_clean", col("AREA").cast("int")) \
        .withColumn("rpt_dist_no_clean", col("Rpt_Dist_No").cast("int")) \
        .withColumn("crime_code_clean", col("Crm_Cd").cast("int")) \
        .withColumn("premis_code_clean", col("Premis_Cd").cast("int")) \
        .withColumn("weapon_code_clean", col("Weapon_Used_Cd").cast("int"))

    # 4. Clean victim age
    df_cleaned = df_cleaned \
        .withColumn(
            "victim_age_clean",
            when(
                (col("Vict_Age").cast("int") >= 0) &
                (col("Vict_Age").cast("int") <= 120),
                col("Vict_Age").cast("int")
            ).otherwise(None)
        )

    # 5. Create age groups
    df_cleaned = df_cleaned \
        .withColumn(
            "age_group",
            when(col("victim_age_clean") < 18, "Under 18")
            .when((col("victim_age_clean") >= 18) & (col("victim_age_clean") < 25), "18-24")
            .when((col("victim_age_clean") >= 25) & (col("victim_age_clean") < 35), "25-34")
            .when((col("victim_age_clean") >= 35) & (col("victim_age_clean") < 45), "35-44")
            .when((col("victim_age_clean") >= 45) & (col("victim_age_clean") < 55), "45-54")
            .when((col("victim_age_clean") >= 55) & (col("victim_age_clean") < 65), "55-64")
            .when(col("victim_age_clean") >= 65, "65+")
            .otherwise("Unknown")
        )

    # 6. Clean victim sex
    df_cleaned = df_cleaned \
        .withColumn(
            "victim_sex_clean",
            when(col("Vict_Sex").isin("M", "F", "X"), col("Vict_Sex"))
            .when(col("Vict_Sex") == "H", "M")
            .when(col("Vict_Sex") == "-", None)
            .otherwise("Unknown")
        )

    # 7. Clean coordinates
    df_cleaned = df_cleaned \
        .withColumn(
            "latitude_clean",
            when(
                (col("`LAT`").cast("decimal(9,6)") != 0) &
                col("`LAT`").cast("decimal(9,6)").isNotNull(),
                col("`LAT`").cast("decimal(9,6)")
            ).otherwise(None)
        ) \
        .withColumn(
            "longitude_clean",
            when(
                (col("`LON`").cast("decimal(9,6)") != 0) &
                col("`LON`").cast("decimal(9,6)").isNotNull(),
                col("`LON`").cast("decimal(9,6)")
            ).otherwise(None)
        )

    # 8. Standardize status
    df_cleaned = df_cleaned \
        .withColumn("status_clean", upper(trim(col("`Status`"))))

    # 9. Create arrest flag
    df_cleaned = df_cleaned \
        .withColumn(
            "arrest_flag",
            when(col("status_clean").like("%ARREST%"), "Y").otherwise("N")
        )

    # 10. Determine arrest type
    df_cleaned = df_cleaned \
        .withColumn(
            "arrest_type",
            when(col("status_clean") == "INVEST CONT", "Investigation Continuing")
            .when(col("status_clean").like("%JUV%"), "Juvenile")
            .when(col("status_clean").like("%ADULT%"), "Adult")
            .when(col("status_clean").like("%OTHER%"), "Other")
            .otherwise(None)
        )

    # 11. Parse Part 1-2 crimes
    df_cleaned = df_cleaned.withColumn(
        "crime_part_clean",
        when(col("`Part_1-2`") == "1", "Part 1")
        .when(col("`Part_1-2`") == "2", "Part 2")
        .otherwise("Unknown")
    )

    # 12. Create crime category based on crime code description
    df_cleaned = df_cleaned \
        .withColumn(
            "crime_category",
            when(col("Crm_Cd_Desc").like("%THEFT%"), "Theft")
            .when(col("Crm_Cd_Desc").like("%ASSAULT%"), "Assault")
            .when(col("Crm_Cd_Desc").like("%BURGLARY%"), "Burglary")
            .when(col("Crm_Cd_Desc").like("%ROBBERY%"), "Robbery")
            .when(col("Crm_Cd_Desc").like("%VANDALISM%"), "Vandalism")
            .when(col("Crm_Cd_Desc").like("%RAPE%"), "Sexual Assault")
            .when(
                col("Crm_Cd_Desc").like("%HOMICIDE%") |
                col("Crm_Cd_Desc").like("%MURDER%"),
                "Homicide"
            )
            .when(col("Crm_Cd_Desc").like("%VEHICLE%"), "Vehicle-Related")
            .when(col("Crm_Cd_Desc").like("%FRAUD%"), "Fraud")
            .when(
                col("Crm_Cd_Desc").like("%DRUG%") |
                col("Crm_Cd_Desc").like("%NARCOTICS%"),
                "Drug-Related"
            )
            .otherwise("Other")
        )

    # 13. Create time period buckets
    df_cleaned = df_cleaned \
        .withColumn(
            "time_period",
            when((col("hour_occurred") >= 6) & (col("hour_occurred") < 12), "Morning")
            .when((col("hour_occurred") >= 12) & (col("hour_occurred") < 18), "Afternoon")
            .when((col("hour_occurred") >= 18) & (col("hour_occurred") < 24), "Evening")
            .when((col("hour_occurred") >= 0) & (col("hour_occurred") < 6), "Night")
            .otherwise("Unknown")
        )

    # 14. Create day of week from date
    df_cleaned = df_cleaned \
        .withColumn("day_of_week", dayofweek(col("date_occurred_clean"))) \
        .withColumn(
            "day_name",
            when(col("day_of_week") == 1, "Sunday")
            .when(col("day_of_week") == 2, "Monday")
            .when(col("day_of_week") == 3, "Tuesday")
            .when(col("day_of_week") == 4, "Wednesday")
            .when(col("day_of_week") == 5, "Thursday")
            .when(col("day_of_week") == 6, "Friday")
            .when(col("day_of_week") == 7, "Saturday")
            .otherwise(None)
        ) \
        .withColumn(
            "is_weekend",
            when(col("day_of_week").isin(1, 7), "Y").otherwise("N")
        )

    # 15. Extract date components
    df_cleaned = df_cleaned \
        .withColumn("year", year(col("date_occurred_clean"))) \
        .withColumn("month", month(col("date_occurred_clean"))) \
        .withColumn("month_name", date_format(col("date_occurred_clean"), "MMMM")) \
        .withColumn("quarter", quarter(col("date_occurred_clean"))) \
        .withColumn("week_of_year", weekofyear(col("date_occurred_clean"))) \
        .withColumn("day_of_month", dayofmonth(col("date_occurred_clean")))

    # 16. Create date keys for dimension tables
    df_cleaned = df_cleaned \
        .withColumn(
            "date_key",
            date_format(col("date_occurred_clean"), "yyyyMMdd").cast("int")
        ) \
        .withColumn(
            "time_key",
            when(col("time_occurred_clean").isNotNull(),
                 col("time_occurred_clean").cast("int"))
            .otherwise(None)
        )

    # 17. Clean location address
    df_cleaned = df_cleaned \
        .withColumn(
            "location_address_clean",
            regexp_replace(trim(col("`LOCATION`")), r"[\r\n]+", " ")
        )

    # 18. Create weapon category
    df_cleaned = df_cleaned.withColumn(
        "weapon_category",
        when(col("Weapon_Desc").isNull() | (col("Weapon_Desc") == ""), "No Weapon")
        .when(
            col("Weapon_Desc").like("%FIREARM%") |
            col("Weapon_Desc").like("%GUN%") |
            col("Weapon_Desc").like("%PISTOL%") |
            col("Weapon_Desc").like("%RIFLE%"),
            "Firearm"
        )
        .when(
            col("Weapon_Desc").like("%KNIFE%") |
            col("Weapon_Desc").like("%BLADE%"),
            "Knife/Blade"
        )
        .when(
            col("Weapon_Desc").like("%HANDS%") |
            col("Weapon_Desc").like("%FIST%") |
            col("Weapon_Desc").like("%FEET%"),
            "Body Parts"
        )
        .when(col("Weapon_Desc").like("%VEHICLE%"), "Vehicle")
        .otherwise("Other Weapon")
    )

    # 19. Victim descent description
    df_cleaned = df_cleaned.withColumn(
        "descent_description",
        when(col("Vict_Descent") == "A", "Asian")
        .when(col("Vict_Descent") == "B", "Black")
        .when(col("Vict_Descent") == "C", "Chinese")
        .when(col("Vict_Descent") == "F", "Filipino")
        .when(col("Vict_Descent") == "G", "Guamanian")
        .when(col("Vict_Descent") == "H", "Hispanic/Latino")
        .when(col("Vict_Descent") == "I", "American Indian")
        .when(col("Vict_Descent") == "J", "Japanese")
        .when(col("Vict_Descent") == "K", "Korean")
        .when(col("Vict_Descent") == "L", "Laotian")
        .when(col("Vict_Descent") == "O", "Other")
        .when(col("Vict_Descent") == "P", "Pacific Islander")
        .when(col("Vict_Descent") == "S", "Samoan")
        .when(col("Vict_Descent") == "U", "Hawaiian")
        .when(col("Vict_Descent") == "V", "Vietnamese")
        .when(col("Vict_Descent") == "W", "White")
        .when(col("Vict_Descent") == "X", "Unknown")
        .when(col("Vict_Descent") == "Z", "Asian Indian")
        .otherwise("Unknown")
    )

    # 20. Add data quality flags
    df_cleaned = df_cleaned \
        .withColumn(
            "has_valid_coordinates",
            when(
                (col("latitude_clean").isNotNull()) &
                (col("longitude_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        ) \
        .withColumn(
            "has_victim_info",
            when(
                (col("victim_age_clean").isNotNull()) &
                (col("victim_sex_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        ) \
        .withColumn(
            "data_quality_score",
            when(col("has_valid_coordinates") == "Y", 1).otherwise(0) +
            when(col("has_victim_info") == "Y", 1).otherwise(0) +
            when(col("Weapon_Desc").isNotNull(), 1).otherwise(0) +
            when(col("Cross_Street").isNotNull(), 1).otherwise(0)
        )

    # =============================
    # SELECT FINAL SILVER COLUMNS
    # =============================

    silver_columns = [
        # Record identifiers
        "dr_no_clean",
        "bronze_record_id",

        # Date and time columns
        "date_reported_clean",
        "date_occurred_clean",
        "time_occurred_clean",
        "hour_occurred",
        "minute_occurred",
        "time_period",
        "date_key",
        "time_key",

        # Date components
        "year",
        "month",
        "month_name",
        "quarter",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",

        # Location columns
        "area_clean",
        col("AREA_NAME").alias("area_name"),
        "rpt_dist_no_clean",
        "premis_code_clean",
        col("Premis_Desc").alias("premis_desc"),
        "location_address_clean",
        col("Cross_Street").alias("cross_street"),
        "latitude_clean",
        "longitude_clean",
        "has_valid_coordinates",

        # Crime columns
        "crime_code_clean",
        col("Crm_Cd_Desc").alias("crime_desc"),
        "crime_part_clean",
        "crime_category",
        col("Mocodes").alias("mocodes"),
        col("Crm_Cd_1").alias("crm_cd_1"),
        col("Crm_Cd_2").alias("crm_cd_2"),
        col("Crm_Cd_3").alias("crm_cd_3"),
        col("Crm_Cd_4").alias("crm_cd_4"),

        # Victim columns
        "victim_age_clean",
        "age_group",
        "victim_sex_clean",
        col("Vict_Descent").alias("victim_descent"),
        "descent_description",
        "has_victim_info",

        # Weapon columns
        "weapon_code_clean",
        col("Weapon_Desc").alias("weapon_desc"),
        "weapon_category",

        # Status columns
        "status_clean",
        col("Status_Desc").alias("status_desc"),
        "arrest_flag",
        "arrest_type",

        # Metadata
        "data_quality_score",
        current_timestamp().alias("silver_processing_timestamp"),
        col("ingestion_timestamp").alias("bronze_ingestion_timestamp")
    ]

    df_silver = df_cleaned.select(*silver_columns)
    df_silver = df_silver.dropDuplicates(["dr_no_clean"])

    # ⛔️ No .write() in DLT — just return the DataFrame
    return df_silver


In [0]:
%skip
def validate_silver_layer(silver_table_name):
    """
    Validate Silver layer data quality
    """
    df_silver = (
        spark.read
        .format("delta")
        .load("/Volumes/medalliaon_replication/la_crime/la_crime")
    )
    
    print("\n" + "=" * 80)
    print("SILVER LAYER VALIDATION REPORT")
    print("=" * 80)
    
    # Record count
    total_records = df_silver.count()
    print(f"\n✅ Total records in Silver layer: {total_records:,}")
    
    # Check for duplicates
    duplicate_count = df_silver.groupBy("dr_no_clean").count().filter("count > 1").count()
    print(f"✅ Duplicate DR_NO records: {duplicate_count}")
    
    # Data quality metrics
    print("\n📊 DATA QUALITY METRICS:")
    
    quality_metrics = df_silver.agg(
        count(when(col("has_valid_coordinates") == "Y", 1)).alias("records_with_coordinates"),
        count(when(col("has_victim_info") == "Y", 1)).alias("records_with_victim_info"),
        count(when(col("arrest_flag") == "Y", 1)).alias("arrest_records"),
        count(when(col("weapon_code_clean").isNotNull(), 1)).alias("records_with_weapon"),
        avg("data_quality_score").alias("avg_quality_score"),
        min("date_occurred_clean").alias("min_date"),
        max("date_occurred_clean").alias("max_date")
    ).collect()[0]
    
    print(f"   Records with valid coordinates: {quality_metrics['records_with_coordinates']:,} ({quality_metrics['records_with_coordinates']/total_records*100:.1f}%)")
    print(f"   Records with victim info: {quality_metrics['records_with_victim_info']:,} ({quality_metrics['records_with_victim_info']/total_records*100:.1f}%)")
    print(f"   Arrest records: {quality_metrics['arrest_records']:,} ({quality_metrics['arrest_records']/total_records*100:.1f}%)")
    print(f"   Records with weapon info: {quality_metrics['records_with_weapon']:,} ({quality_metrics['records_with_weapon']/total_records*100:.1f}%)")
    print(f"   Average data quality score: {quality_metrics['avg_quality_score']:.2f}/4")
    print(f"   Date range: {quality_metrics['min_date']} to {quality_metrics['max_date']}")
    
    # Crime category distribution
    print("\n🔍 CRIME CATEGORY DISTRIBUTION:")
    crime_dist = df_silver.groupBy("crime_category") \
        .count() \
        .orderBy(desc("count")) \
        .limit(10)
    
    crime_dist.show()
    
    # Time period distribution
    print("\n⏰ TIME PERIOD DISTRIBUTION:")
    time_dist = df_silver.groupBy("time_period") \
        .count() \
        .orderBy("time_period")
    
    time_dist.show()
    
    return df_silver

# Validate Silver layer
validate_silver_layer("la_crime_clean")


In [0]:
%skip
# ===========================
# SILVER → SNOWFLAKE GOLD LOAD
# (Run this as ONE cell in Databricks)
# ===========================

from pyspark.sql import functions as F

# 1️⃣ Snowflake connection options (EDIT THESE)
sfOptions = {
    "sfURL": "<ACCOUNT>.snowflakecomputing.com",          # e.g. "xy12345.us-east-1.snowflakecomputing.com"
    "sfUser": "<USERNAME>",
    "sfPassword": dbutils.secrets.get("sf_scope", "sf_password"),  # or put plain text for quick testing
    "sfDatabase": "LA_CRIME_DW",
    "sfSchema": "GOLD",           # default schema (GOLD)
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "SYSADMIN"          # or your role
}

# 2️⃣ Read SILVER table from Delta
silver_df = (
    spark.read
         .format("delta")
         .load("/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/silver/la_crime_clean")
)

# 3️⃣ Build GOLD DIMENSION dataframes

# DIM_DATE
dim_date_df = (
    silver_df
      .select(
          F.col("date_key").cast("int").alias("date_key"),
          F.col("date_occurred_clean").alias("full_date"),
          F.col("day_of_week").cast("int").alias("day_of_week"),
          F.col("day_name"),
          F.col("day_of_month").cast("int").alias("day_of_month"),
          F.col("week_of_year").cast("int").alias("week_of_year"),
          F.col("month").cast("int").alias("month"),
          F.col("month_name"),
          F.col("quarter").cast("int").alias("quarter"),
          F.col("year").cast("int").alias("year"),
          F.col("is_weekend")
      )
      .dropDuplicates(["date_key"])
)

# DIM_TIME
dim_time_df = (
    silver_df
      .withColumn(
          "time_occ",
          F.to_timestamp(
              F.concat_ws(
                  ":",
                  F.lit("1970-01-01"),
                  F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                  F.lpad(F.col("minute_occurred").cast("string"), 2, "0")
              ),
              "yyyy-MM-dd:HH:mm"
          ).cast("time")
      )
      .withColumn(
          "time_bucket",
          F.when(
              F.col("hour_occurred").isNotNull(),
              F.concat(
                  F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                  F.lit(":00–"),
                  F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                  F.lit(":59")
              )
          )
      )
      .select(
          F.col("time_key").cast("int").alias("time_key"),
          F.col("time_occ"),
          F.col("hour_occurred").cast("int").alias("hour"),
          F.col("time_period"),
          F.col("time_bucket")
      )
      .dropDuplicates(["time_key"])
)

# DIM_CRIME
dim_crime_df = (
    silver_df
      .select(
          F.col("crime_code_clean").cast("int").alias("crm_cd"),
          F.col("crime_desc").alias("crm_cd_desc"),
          F.col("crime_part_clean").alias("part_1_2"),
          F.col("crime_category"),
          F.col("mocodes")
      )
      .dropDuplicates(["crm_cd"])
)

# DIM_VICTIM
dim_victim_df = (
    silver_df
      .select(
          F.col("victim_age_clean").cast("int").alias("vict_age"),
          F.col("age_group"),
          F.col("victim_sex_clean").alias("vict_sex"),
          F.col("victim_descent"),
          F.col("descent_description")
      )
      .dropDuplicates(["vict_age", "vict_sex", "vict_descent"])
)

# DIM_STATUS
dim_status_df = (
    silver_df
      .withColumn(
          "case_resolution_type",
          F.when(F.col("arrest_flag") == "Y", "Arrested")
           .when(F.upper(F.col("status_clean")).like("%INVEST%"), "Investigation Continuing")
           .when(F.upper(F.col("status_clean")).like("%JUV%"), "Juvenile Case")
           .when(F.upper(F.col("status_clean")).like("%ADULT%"), "Adult Case")
           .otherwise("Other/Unknown")
      )
      .select(
          F.col("status_clean").alias("status"),
          F.col("status_desc"),
          F.col("arrest_flag").alias("is_arrest"),
          F.col("case_resolution_type")
      )
      .dropDuplicates(["status", "status_desc", "is_arrest"])
)

# DIM_WEAPON
dim_weapon_df = (
    silver_df
      .select(
          F.col("weapon_code_clean").cast("int").alias("weapon_used_cd"),
          F.col("weapon_desc"),
          F.col("weapon_category")
      )
      .dropDuplicates(["weapon_used_cd", "weapon_desc", "weapon_category"])
)

# STAGING.STG_LOCATION (for DIM_LOCATION merge)
stg_location_df = (
    silver_df
      .select(
          F.col("area_clean").cast("int").alias("area"),
          F.col("area_name"),
          F.col("rpt_dist_no_clean").cast("int").alias("rpt_dist_no"),
          F.col("premis_code_clean").cast("int").alias("premis_cd"),
          F.col("premis_desc"),
          F.col("location_address_clean").alias("location_address"),
          F.col("cross_street"),
          F.col("latitude_clean").alias("lat"),
          F.col("longitude_clean").alias("lon"),
          F.lit("N").alias("crime_hotspot_flag"),
          F.lit("LOW").alias("crime_severity_zone")
      )
      .dropDuplicates(["area", "rpt_dist_no", "premis_cd", "location_address"])
)

# 4️⃣ Helper to write to Snowflake
def write_to_snowflake(df, table_name, mode="overwrite", schema=None):
    options = sfOptions.copy()
    if schema:
        options["sfSchema"] = schema
    (
        df.write
          .format("snowflake")
          .options(**options)
          .option("dbtable", table_name)
          .mode(mode)
          .save()
    )

# 5️⃣ WRITE ALL TABLES TO SNOWFLAKE

# GOLD dimensions
write_to_snowflake(dim_date_df,   "DIM_DATE",   mode="overwrite", schema="GOLD")
write_to_snowflake(dim_time_df,   "DIM_TIME",   mode="overwrite", schema="GOLD")
write_to_snowflake(dim_crime_df,  "DIM_CRIME",  mode="overwrite", schema="GOLD")
write_to_snowflake(dim_victim_df, "DIM_VICTIM", mode="overwrite", schema="GOLD")
write_to_snowflake(dim_status_df, "DIM_STATUS", mode="overwrite", schema="GOLD")
write_to_snowflake(dim_weapon_df, "DIM_WEAPON", mode="overwrite", schema="GOLD")

# STAGING location (for MERGE into DIM_LOCATION in Snowflake)
write_to_snowflake(stg_location_df, "STG_LOCATION", mode="overwrite", schema="STAGING")

print("✅ Load to Snowflake GOLD + STAGING completed. Now run the MERGE for DIM_LOCATION in Snowflake.")



In [0]:
%skip
from pyspark.sql import functions as F

# ----------------------------------------------------
# 1. SNOWFLAKE CONNECTION (EDIT THESE)
# ----------------------------------------------------
sfOptions = {
    "sfURL": "RKEDNXQ-INA32978.snowflakecomputing.com",
    "sfUser": "nagar",
    "sfPassword": "LAcrime01234$#",
    "sfDatabase": "LA_CRIME_DW",
    "sfSchema": "GOLD",
    "sfWarehouse": "COMPUTE_WH"
}

# ----------------------------------------------------
# 2. LOAD SILVER TABLE (DLT SAFE)
# ----------------------------------------------------
# ❗ USE spark.table(), NOT .format("delta")
silver_df = spark.table("workspace.default.la_crime_clean")

# ----------------------------------------------------
# 3. DIM_DATE
# ----------------------------------------------------
dim_date_df = (
    silver_df
      .select(
          F.col("date_key").cast("int"),
          F.col("date_occurred_clean").alias("full_date"),
          F.col("day_of_week").cast("int"),
          "day_name",
          F.col("day_of_month"),
          F.col("week_of_year"),
          F.col("month"),
          "month_name",
          F.col("quarter"),
          F.col("year"),
          "is_weekend"
      ).dropDuplicates(["date_key"])
)

# ----------------------------------------------------
# 4. DIM_TIME
# ----------------------------------------------------
dim_time_df = (
    silver_df
      .withColumn(
          "time_occ",
          F.to_timestamp(
              F.concat_ws(
                  ":",
                  F.lit("1970-01-01"),
                  F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                  F.lpad(F.col("minute_occurred").cast("string"), 2, "0")
              ),
              "yyyy-MM-dd:HH:mm"
          )  # Removed .cast("time")
      )
      .withColumn(
          "time_bucket",
          F.concat(
              F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
              F.lit(":00–"),
              F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
              F.lit(":59")
          )
      )
      .select(
          F.col("time_key").cast("int"),
          "time_occ",
          F.col("hour_occurred").alias("hour"),
          "time_period",
          "time_bucket"
      ).dropDuplicates(["time_key"])
)

# ----------------------------------------------------
# 5. DIM_CRIME
# ----------------------------------------------------
dim_crime_df = (
    silver_df
      .select(
          F.col("crime_code_clean").alias("crm_cd"),
          F.col("crime_desc").alias("crm_cd_desc"),
          F.col("crime_part_clean").alias("part_1_2"),
          "crime_category",
          "mocodes"
      ).dropDuplicates(["crm_cd"])
)

# ----------------------------------------------------
# 6. DIM_VICTIM
# ----------------------------------------------------
# 6️⃣ DIM_VICTIM
# 6️⃣ DIM_VICTIM
dim_victim_df = (
    silver_df
      .select(
          F.col("victim_age_clean").cast("int").alias("vict_age"),
          F.col("age_group"),
          F.col("victim_sex_clean").alias("vict_sex"),
          F.col("victim_descent"),          # <-- correct column name from Silver
          F.col("descent_description")
      )
      # use the *output* column names here
      .dropDuplicates(["vict_age", "vict_sex", "victim_descent"])
)

# ----------------------------------------------------
# 7. DIM_STATUS
# ----------------------------------------------------
dim_status_df = (
    silver_df
      .withColumn(
          "case_resolution_type",
          F.when(F.col("arrest_flag")=="Y","Arrested")
           .when(F.upper(F.col("status_clean")).like("%INVEST%"),"Investigation Continuing")
           .when(F.upper(F.col("status_clean")).like("%JUV%"),"Juvenile Case")
           .when(F.upper(F.col("status_clean")).like("%ADULT%"),"Adult Case")
           .otherwise("Other/Unknown")
      )
      .select(
          F.col("status_clean").alias("status"),
          "status_desc",
          F.col("arrest_flag").alias("is_arrest"),
          "case_resolution_type"
      ).dropDuplicates(["status","status_desc","is_arrest"])
)

# ----------------------------------------------------
# 8. DIM_WEAPON
# ----------------------------------------------------
dim_weapon_df = (
    silver_df
      .select(
          F.col("weapon_code_clean").alias("weapon_used_cd"),
          "weapon_desc",
          "weapon_category"
      ).dropDuplicates(["weapon_used_cd","weapon_desc","weapon_category"])
)

# ----------------------------------------------------
# 9. STG_LOCATION (for DIM_LOCATION MERGE)
# ----------------------------------------------------
stg_location_df = (
    silver_df
      .select(
          F.col("area_clean").alias("area"),
          "area_name",
          F.col("rpt_dist_no_clean").alias("rpt_dist_no"),
          F.col("premis_code_clean").alias("premis_cd"),
          "premis_desc",
          F.col("location_address_clean").alias("location_address"),
          "cross_street",
          F.col("latitude_clean").alias("lat"),
          F.col("longitude_clean").alias("lon"),
          F.lit("N").alias("crime_hotspot_flag"),
          F.lit("LOW").alias("crime_severity_zone")
      ).dropDuplicates(["area","rpt_dist_no","premis_cd","location_address"])
)

# ----------------------------------------------------
# 10. WRITE FUNCTION
# ----------------------------------------------------
def write_to_snowflake(df, table, schema):
    opts = sfOptions.copy()
    opts["sfSchema"] = schema
    (
        df.write.format("snowflake")
          .options(**opts)
          .option("dbtable", table)
          .mode("overwrite")
          .save()
    )

# ----------------------------------------------------
# 11. PUSH TO SNOWFLAKE
# ----------------------------------------------------
write_to_snowflake(dim_date_df,   "DIM_DATE",   "GOLD")
write_to_snowflake(dim_time_df,   "DIM_TIME",   "GOLD")
write_to_snowflake(dim_crime_df,  "DIM_CRIME",  "GOLD")
write_to_snowflake(dim_victim_df, "DIM_VICTIM", "GOLD")
write_to_snowflake(dim_status_df, "DIM_STATUS", "GOLD")
write_to_snowflake(dim_weapon_df, "DIM_WEAPON", "GOLD")

# staging for DIM_LOCATION
write_to_snowflake(stg_location_df, "STG_LOCATION", "STAGING")

print("✅ GOLD DIMENSIONS + STAGING LOCATION pushed successfully.")


In [0]:
%skip
import dlt
from pyspark.sql import functions as F

# ======================================================
# 1. Read Silver inside DLT using dlt.read()
# ======================================================

@dlt.table(
    name="gold_la_crime_date_dim",
    comment="Date dimension for LA Crime"
)
def gold_dim_date():
    df = dlt.read("la_crime_clean")  # Your Silver table
    return (
        df.select(
            F.col("date_key").cast("int").alias("date_key"),
            F.col("date_occurred_clean").alias("full_date"),
            F.col("day_of_week").cast("int"),
            "day_name",
            F.col("day_of_month"),
            F.col("week_of_year"),
            F.col("month"),
            "month_name",
            F.col("quarter"),
            F.col("year"),
            "is_weekend"
        ).dropDuplicates(["date_key"])
    )

@dlt.table(
    name="gold_la_crime_time_dim",
    comment="Time dimension"
)
def gold_dim_time():
    df = dlt.read("la_crime_clean")
    return (
        df.withColumn(
            "time_occ",
            F.concat_ws(
                ":",
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lpad(F.col("minute_occurred").cast("string"), 2, "0")
            )
        )
        .withColumn(
            "time_bucket",
            F.concat(
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lit(":00–"),
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lit(":59")
            )
        )
        .select(
            F.col("time_key").cast("int").alias("time_key"),
            "time_occ",
            F.col("hour_occurred").alias("hour"),
            "time_period",
            "time_bucket"
        )
        .dropDuplicates(["time_key"])
    )

@dlt.table(name="gold_la_crime_dim_crime")
def gold_dim_crime():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("crime_code_clean").alias("crm_cd"),
            F.col("crime_desc").alias("crm_cd_desc"),
            F.col("crime_part_clean").alias("part_1_2"),
            "crime_category",
            "mocodes"
        ).dropDuplicates(["crm_cd"])
    )

@dlt.table(name="gold_la_crime_dim_victim")
def gold_dim_victim():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("victim_age_clean").cast("int").alias("vict_age"),
            "age_group",
            F.col("victim_sex_clean").alias("vict_sex"),
            "victim_descent",
            "descent_description"
        ).dropDuplicates(["vict_age","vict_sex","victim_descent"])
    )

@dlt.table(name="gold_la_crime_dim_status")
def gold_dim_status():
    df = dlt.read("la_crime_clean")
    return (
        df.withColumn(
            "case_resolution_type",
            F.when(F.col("arrest_flag")=="Y","Arrested")
             .when(F.upper(F.col("status_clean")).like("%INVEST%"),"Investigation Continuing")
             .when(F.upper(F.col("status_clean")).like("%JUV%"),"Juvenile Case")
             .when(F.upper(F.col("status_clean")).like("%ADULT%"),"Adult Case")
             .otherwise("Other/Unknown")
        )
        .select(
            F.col("status_clean").alias("status"),
            "status_desc",
            F.col("arrest_flag").alias("is_arrest"),
            "case_resolution_type"
        )
        .dropDuplicates(["status","status_desc","is_arrest"])
    )

@dlt.table(name="gold_la_crime_dim_weapon")
def gold_dim_weapon():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("weapon_code_clean").alias("weapon_used_cd"),
            "weapon_desc",
            "weapon_category"
        ).dropDuplicates(["weapon_used_cd","weapon_desc","weapon_category"])
    )

@dlt.table(name="gold_la_crime_stg_location")
def gold_stg_location():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("area_clean").alias("area"),
            "area_name",
            F.col("rpt_dist_no_clean").alias("rpt_dist_no"),
            F.col("premis_code_clean").alias("premis_cd"),
            "premis_desc",
            F.col("location_address_clean").alias("location_address"),
            "cross_street",
            F.col("latitude_clean").alias("lat"),
            F.col("longitude_clean").alias("lon"),
            F.lit("N").alias("crime_hotspot_flag"),
            F.lit("LOW").alias("crime_severity_zone")
        ).dropDuplicates(["area","rpt_dist_no","premis_cd","location_address"])
    )


In [0]:
import dlt
from pyspark.sql import functions as F

# ============================================================
# GOLD DIMENSIONS FROM SILVER (la_crime_clean)
# ============================================================

# 1. DATE DIMENSION (Type 1 / snapshot)
@dlt.table(
    name="gold_la_crime_date_dim",
    comment="Date dimension for LA crime"
)
def gold_dim_date():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("date_key").cast("int").alias("date_key"),
            F.col("date_occurred_clean").alias("full_date"),
            F.col("day_of_week").cast("int").alias("day_of_week"),
            F.col("day_name"),
            F.col("day_of_month"),
            F.col("week_of_year"),
            F.col("month"),
            F.col("month_name"),
            F.col("quarter"),
            F.col("year"),
            F.col("is_weekend")
        )
        .dropDuplicates(["date_key"])
    )

# 2. TIME DIMENSION (Type 1 / snapshot)
@dlt.table(
    name="gold_la_crime_time_dim",
    comment="Time dimension for LA crime"
)
def gold_dim_time():
    df = dlt.read("la_crime_clean")
    return (
        df.withColumn(
            "time_occ",
            F.concat_ws(
                ":",
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lpad(F.col("minute_occurred").cast("string"), 2, "0")
            )
        )
        .withColumn(
            "time_bucket",
            F.concat(
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lit(":00–"),
                F.lpad(F.col("hour_occurred").cast("string"), 2, "0"),
                F.lit(":59")
            )
        )
        .select(
            F.col("time_key").cast("int").alias("time_key"),
            F.col("time_occ"),
            F.col("hour_occurred").alias("hour"),
            F.col("time_period"),
            F.col("time_bucket")
        )
        .dropDuplicates(["time_key"])
    )

# 3. CRIME DIMENSION (Type 1)
@dlt.table(
    name="gold_la_crime_dim_crime",
    comment="Crime dimension (snapshot)"
)
def gold_dim_crime():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("crime_code_clean").alias("crm_cd"),
            F.col("crime_desc").alias("crm_cd_desc"),
            F.col("crime_part_clean").alias("part_1_2"),
            F.col("crime_category"),
            F.col("mocodes")
        )
        .dropDuplicates(["crm_cd"])
    )

# 4. VICTIM DIMENSION (Type 1)
@dlt.table(
    name="gold_la_crime_dim_victim",
    comment="Victim dimension (snapshot)"
)
def gold_dim_victim():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("victim_age_clean").cast("int").alias("vict_age"),
            F.col("age_group"),
            F.col("victim_sex_clean").alias("vict_sex"),
            F.col("victim_descent"),
            F.col("descent_description")
        )
        .dropDuplicates(["vict_age", "vict_sex", "victim_descent"])
    )

# 5. WEAPON DIMENSION (Type 1)
@dlt.table(
    name="gold_la_crime_dim_weapon",
    comment="Weapon dimension (snapshot)"
)
def gold_dim_weapon():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("weapon_code_clean").alias("weapon_used_cd"),
            F.col("weapon_desc"),
            F.col("weapon_category")
        )
        .dropDuplicates(["weapon_used_cd", "weapon_desc", "weapon_category"])
    )

# 6. LOCATION STAGING DIM (Type 1, used later for SCD2 in Snowflake or another Delta table)
@dlt.table(
    name="gold_la_crime_stg_location",
    comment="Location staging dimension (snapshot); can be used for SCD2 merges downstream"
)
def gold_stg_location():
    df = dlt.read("la_crime_clean")
    return (
        df.select(
            F.col("area_clean").alias("area"),
            F.col("area_name"),
            F.col("rpt_dist_no_clean").alias("rpt_dist_no"),
            F.col("premis_code_clean").alias("premis_cd"),
            F.col("premis_desc"),
            F.col("location_address_clean").alias("location_address"),
            F.col("cross_street"),
            F.col("latitude_clean").alias("lat"),
            F.col("longitude_clean").alias("lon"),
            F.lit("N").alias("crime_hotspot_flag"),
            F.lit("LOW").alias("crime_severity_zone")
        )
        .dropDuplicates(["area", "rpt_dist_no", "premis_cd", "location_address"])
    )

# ============================================================
# 7. STATUS DIMENSION – SCD TYPE 2 IMPLEMENTATION
# ============================================================

@dlt.table(
    name="gold_la_crime_dim_status_scd2",
    comment="Status dimension with SCD Type 2 (tracks history of status / arrest_flag / description)"
)
def gold_dim_status_scd2_initial_load():
    """
    For the first load, we create a full SCD2 dimension from the Silver data:
      - status_key          : surrogate key (generated in Databricks)
      - status              : natural key (from status_clean)
      - status_desc         : description
      - is_arrest           : Y/N
      - case_resolution_type: derived category
      - effective_start_dt  : when this version became active
      - effective_end_dt    : end of validity (9999-12-31 for current)
      - is_current          : 'Y' for current, 'N' for old versions

    NOTE: This function assumes an initial full load.
    For incremental daily loads, you would MERGE new
    snapshots into this table and:
      - close out old versions (set effective_end_dt, is_current = 'N')
      - insert new rows for changed records.
    """
    df = dlt.read("la_crime_clean")

    # derive case_resolution_type
    df_status = (
        df.withColumn(
            "case_resolution_type",
            F.when(F.col("arrest_flag") == "Y", "Arrested")
             .when(F.upper(F.col("status_clean")).like("%INVEST%"), "Investigation Continuing")
             .when(F.upper(F.col("status_clean")).like("%JUV%"), "Juvenile Case")
             .when(F.upper(F.col("status_clean")).like("%ADULT%"), "Adult Case")
             .otherwise("Other/Unknown")
        )
        .select(
            F.col("status_clean").alias("status"),
            F.col("status_desc"),
            F.col("arrest_flag").alias("is_arrest"),
            F.col("case_resolution_type")
        )
        .dropDuplicates(["status", "status_desc", "is_arrest"])
    )

    # add SCD2 metadata columns
    current_dt = F.current_date()
    max_dt = F.to_date(F.lit("9999-12-31"))

    scd2_df = (
        df_status
        .withColumn("status_key", F.monotonically_increasing_id())
        .withColumn("effective_start_dt", current_dt)
        .withColumn("effective_end_dt", max_dt)
        .withColumn("is_current", F.lit("Y"))
    )

    # reorder columns
    scd2_df = scd2_df.select(
        "status_key",
        "status",
        "status_desc",
        "is_arrest",
        "case_resolution_type",
        "effective_start_dt",
        "effective_end_dt",
        "is_current"
    )

    return scd2_df


In [0]:
import dlt
from pyspark.sql import functions as F

@dlt.table(
    name="gold_la_crime_fact_incidents",
    comment="Fact table of LA crime incidents at incident grain (DR_NO)"
)
def gold_fact_crime_incidents():
    # Silver source
    s = dlt.read("la_crime_clean")

    # Bring in dimension tables (DLT Gold)
    dim_date     = dlt.read("gold_la_crime_date_dim")
    dim_time     = dlt.read("gold_la_crime_time_dim")
    dim_crime    = dlt.read("gold_la_crime_dim_crime")
    dim_victim   = dlt.read("gold_la_crime_dim_victim")
    dim_status   = dlt.read("gold_la_crime_dim_status")
    dim_weapon   = dlt.read("gold_la_crime_dim_weapon")
    dim_location = dlt.read("gold_la_crime_stg_location")   # acts as location dim in Databricks

    # Join to dimensions on natural keys
    fact = (
        s.alias("s")
        # Date
        .join(dim_date.alias("dd"), F.col("s.date_key") == F.col("dd.date_key"), "left")
        # Time
        .join(dim_time.alias("dt"), F.col("s.time_key") == F.col("dt.time_key"), "left")
        # Crime
        .join(dim_crime.alias("dc"),
              F.col("s.crime_code_clean") == F.col("dc.crm_cd"),
              "left")
        # Victim
        .join(dim_victim.alias("dv"),
              (F.col("s.victim_age_clean") == F.col("dv.vict_age")) &
              (F.col("s.age_group")        == F.col("dv.age_group")) &
              (F.col("s.victim_sex_clean") == F.col("dv.vict_sex")) &
              (F.col("s.victim_descent")   == F.col("dv.victim_descent")),
              "left")
        # Status
        .join(dim_status.alias("ds"),
              (F.col("s.status_clean") == F.col("ds.status")) &
              (F.col("s.status_desc")  == F.col("ds.status_desc")) &
              (F.col("s.arrest_flag")  == F.col("ds.is_arrest")),
              "left")
        # Weapon (left join because some incidents have no weapon)
        .join(dim_weapon.alias("dw"),
              (F.col("s.weapon_code_clean") == F.col("dw.weapon_used_cd")) &
              (F.col("s.weapon_desc")       == F.col("dw.weapon_desc")),
              "left")
        # Location
        .join(dim_location.alias("dl"),
              (F.col("s.area_clean")              == F.col("dl.area")) &
              (F.col("s.rpt_dist_no_clean")       == F.col("dl.rpt_dist_no")) &
              (F.col("s.premis_code_clean")       == F.col("dl.premis_cd")) &
              (F.col("s.location_address_clean")  == F.col("dl.location_address")),
              "left")
    )

    # Build final fact projection
    fact_final = fact.select(
        F.col("s.dr_no_clean").alias("dr_no"),
        F.col("s.date_key").alias("date_key"),
        F.col("s.time_key").alias("time_key"),
        # In Databricks DLT, dim tables don't yet have surrogate IDs,
        # so we just reuse natural keys / or generate IDs later in Snowflake.
        F.col("dl.area").alias("location_nk_area"),
        F.col("dl.rpt_dist_no").alias("location_nk_rpt_dist_no"),
        F.col("dl.premis_cd").alias("location_nk_premis_cd"),
        F.col("dl.location_address").alias("location_nk_address"),

        F.col("dc.crm_cd").alias("crime_nk_crm_cd"),
        F.col("dv.vict_age").alias("victim_nk_age"),
        F.col("dv.vict_sex").alias("victim_nk_sex"),
        F.col("dv.victim_descent").alias("victim_nk_descent"),
        F.col("dw.weapon_used_cd").alias("weapon_nk_used_cd"),
        F.col("ds.status").alias("status_nk_status"),

        F.col("s.date_reported_clean").alias("date_reported"),
        F.col("s.arrest_flag"),
        F.col("s.arrest_type")
    ).dropDuplicates(["dr_no"])

    return fact_final
